# S-03: Driver-availability ablation (isolating scenario-mode feature-degradation cost from extrapolation cost)

**Supervisor request** (Prof. Paul Harris, relayed by the user): isolate how much forecasting accuracy is lost
purely from NOT having access to real-time sensor variables that a CMIP6 climate scenario can never supply --
distinct from two effects already tested in this project:

- **U-03 (D-63)**: tests whether B-10's models/calibration hold up under *distribution shift* (out-of-envelope
  `fx_lsu_dens` perturbation). Real historical anchors, but the "shock" is an artificially extreme covariate
  value, not a feature-set change.
- **S-01 (D-64)**: builds the actual scenario pipeline (CMIP6 climate + historical-day-resampled drivers +
  livestock multiplier) but evaluates on **unscored 2041-2060 scenario data** -- there is no real ground truth,
  so S-01 cannot separate "the model got worse because the drivers are degraded" from "the model got worse
  because 2050 is genuinely outside the training envelope."

**This experiment fixes that conflation** by holding test data **real and historical** (same 2018-2022 anchors
as B-10/D-65) and only changing the **feature set** -- giving a clean, isolated number for "what scenario-mode
costs before extrapolation even enters the picture."

- **Model 1** (existing, NOT rerun here): B-10's unweighted RF+XGB+LightGBM+SARIMAX ensemble, full feature set,
  historical data. Read directly from `results/b10_b13_rerun_table_all_towers.csv` /
  `_by_tower_year.csv` (D-65).
- **Model 2** (new, this notebook): identical B-10 architecture/hyperparameters/ensemble, same 5-anchor x
  3-tower sweep, two variants:
  - **Variant A (removal)**: scenario-unavailable columns dropped entirely -- model never sees them, training
    or test.
  - **Variant B (resample)**: same columns present and used in training (real values, identical to Model 1's
    own training), but their values in the rollout-time `fx_frame`/`exog` (the 365-day target window) are
    day-of-year-climatology-resampled via `rr.doy_climatology()` -- reusing S-01's own function.
- **Explicitly excluded this round**: S-02's RF-proxy reconstructions (D-69) -- a legitimate follow-up, not
  mixed into this clean two-arm comparison.


## Resolved: exact variable list

Direct inspection confirmed `forecast_daily_v2.csv` has exactly 34 `fx_` columns, and
`src/features/build_scenario_drivers.py` (S-01's actual production code) already partitions every
non-CMIP6-derivable driver into two lists:

```python
RESAMPLED_COLS = ["fx_WS_mean","fx_VPD_mean","fx_RN_mean","fx_PPFD_mean","fx_SWC_mean","fx_TS_mean",
  "fx_wd_sin","fx_wd_cos","fx_SWC_lag7","fx_TS_lag7","fx_SWC_lag14","fx_TS_lag14","fx_SWC_lag21",
  "fx_TS_lag21","fx_SWC_lag28","fx_TS_lag28","fx_SWC_roll7","fx_TS_roll7","fx_SWC_roll14",
  "fx_TS_roll14","fx_grazing_active","fx_days_since_grazing"]   # 22 columns
DROPPED_COLS = ["fx_USTAR_mean", "fx_SHF_mean"]                  # 2 columns
```

**PPFD/RN ambiguity resolved**: `fx_PPFD_mean`/`fx_RN_mean` ARE in `RESAMPLED_COLS` -- S-01 already treats them
exactly like WS/VPD/SWC/TS (historical-day-resampled), not as "kept real." No real S-01-vs-S-02 contradiction
once the actual code is read.

**User-confirmed**: wind direction (`fx_wd_sin`/`fx_wd_cos`) and grazing features
(`fx_grazing_active`/`fx_days_since_grazing`) are **included** -- the final degraded-column list is exactly
`RESAMPLED_COLS + DROPPED_COLS` (24 columns), imported directly from `build_scenario_drivers.py`, not retyped.

**Explicitly OUT of scope** (stay real/untouched in both Model 2 variants):
- `fx_lsu_dens` -- the scenario *lever* S-01 deliberately manipulates, not a missing-sensor variable.
- AR features (`ar_ch4_dlag*`, `ar_ch4_drm7`, `ar_fc_dlag1`) -- S-01 resamples these too, but because no real
  recent CH4/FCO2 history exists in a genuinely blind 2050 future (a temporal-extrapolation problem, not a
  driver-source problem). This experiment evaluates on real historical anchors, where real recent AR history
  genuinely exists -- touching it would reintroduce the exact extrapolation-conflation this experiment exists
  to avoid.

**Customizable by design**: `s03_driver_availability_ablation.py`'s `main()` takes `remove_cols`/
`resample_cols` as independent, overridable parameters (both default to the 24-column list above). This
notebook's default run uses the full default list for both; the cell below shows how to run a narrower
sensitivity check (e.g. resampling only soil variables) without touching the script.


In [ ]:
import sys, os, warnings
warnings.filterwarnings("ignore")
ROOT = r"c:\Users\Nicholas\Documents\COMP0191 MSc Artificial Intelligence for Sustainable Development Project"
sys.path.insert(0, ROOT)
sys.path.insert(0, ROOT + r"\src")
sys.path.insert(0, ROOT + r"\src\features")
sys.path.insert(0, ROOT + r"\notebooks\07_scenario_analysis")

import pandas as pd
import numpy as np

import s03_driver_availability_ablation as s03

print("Default degraded columns:", len(s03.DEFAULT_DEGRADED_COLS))
for c in s03.DEFAULT_DEGRADED_COLS:
    print(" ", c)


Default degraded columns: 24
  fx_WS_mean
  fx_VPD_mean
  fx_RN_mean
  fx_PPFD_mean
  fx_SWC_mean
  fx_TS_mean
  fx_wd_sin
  fx_wd_cos
  fx_SWC_lag7
  fx_TS_lag7
  fx_SWC_lag14
  fx_TS_lag14
  fx_SWC_lag21
  fx_TS_lag21
  fx_SWC_lag28
  fx_TS_lag28
  fx_SWC_roll7
  fx_TS_roll7
  fx_SWC_roll14
  fx_TS_roll14
  fx_grazing_active
  fx_days_since_grazing
  fx_USTAR_mean
  fx_SHF_mean


## Customization example (not executed by default -- illustrates the parameterization)

To run a narrower sensitivity check -- e.g. resample only soil moisture/temperature while Variant A still
drops the full default set -- call `main()` directly with overrides and a `run_label` so outputs land in
separate, non-clobbering files:

```python
s03.main(resample_cols=["fx_SWC_mean", "fx_TS_mean"], run_label="soilonly")
# -> results/s03_summary_soilonly.csv, results/s03_summary_vs_gapfilled_soilonly.csv, results/s03_chains_soilonly.csv
```

`remove_cols` and `resample_cols` are independent -- e.g. `remove_cols=s03.DEFAULT_DEGRADED_COLS,
resample_cols=["fx_WS_mean"]` drops everything in Variant A but only resamples wind speed in Variant B,
leaving every other degraded column real in Variant B. Not run in this notebook (the default full-list run
is the primary deliverable); left here as a template for follow-up sensitivity work.


## Smoke test (one tower, one anchor, both variants) -- before trusting the full sweep

Verifies: Variant A produces a visibly smaller feature matrix; Variant B's climatology-substituted `fx_frame`
differs from real values only in the target degraded columns, only for target-window dates (not training
dates); pre-anchor-only climatology (no leakage); numbers are physically plausible (not NaN/exploded).

In [ ]:
_orig_towers, _orig_anchors = s03.TOWERS, s03.ANCHOR_YEARS
s03.TOWERS, s03.ANCHOR_YEARS = [4], [2021]
out_smoke, out_gf_smoke, chains_smoke = s03.main(run_label="smoketest")
s03.TOWERS, s03.ANCHOR_YEARS = _orig_towers, _orig_anchors

print()
print(out_smoke.groupby(["variant", "model"])[["R2", "MASE", "n"]].mean().round(3))


[S-03] FX_B=34, remove_cols=24, resample_cols=24, FX_A (removal, remaining)=10
[S-03] EXOG_B=8, EXOG_A (removal, remaining)=['fx_lsu_dens', 'fx_DOY_sin', 'fx_DOY_cos', 'fx_is_growing'], exog_resample=['fx_WS_mean', 'fx_VPD_mean', 'fx_USTAR_mean', 'fx_PPFD_mean']

Anchor 2021


  Pooled trees fit, both variants (2s)


C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


...

C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


  Tower 4 done, both variants (38s)
  Anchor 2021 total (40s)

[OK] Saved s03_summary_smoketest.csv (72 rows)
[OK] Saved s03_summary_vs_gapfilled_smoketest.csv (72 rows)
[OK] Saved s03_chains_smoketest.csv (730 rows)

                                     R2   MASE       n
variant    model                                      
A_removal  Ensemble_MASEweighted -0.143  1.014  56.833
           Ensemble_unweighted   -0.142  1.014  56.833
           LightGBM              -0.179  1.029  56.833
           RF                    -0.239  1.055  56.833
           SARIMAX               -0.104  0.983  56.833
           XGB                   -0.186  1.027  56.833
B_resample Ensemble_MASEweighted  0.028  0.908  56.833
           Ensemble_unweighted    0.027  0.909  56.833
           LightGBM              -0.039  0.922  56.833
           RF                    -0.271  1.020  56.833
           SARIMAX                0.034  0.909  56.833
           XGB                   -0.023  0.910  56.833


C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [3]:
# Verification: Variant A's feature matrix is smaller than Variant B's (10 remaining fx_ cols vs 34)
dv = pd.read_csv(f"{s03.HOURLY}/forecast_daily_v2.csv", low_memory=False)
FX_B = [c for c in dv.columns if c.startswith("fx")]
FX_A = [c for c in FX_B if c not in s03.DEFAULT_DEGRADED_COLS]
print(f"Variant A (removal) feature count: {len(FX_A)} fx_ columns")
print(f"Variant B (resample) feature count: {len(FX_B)} fx_ columns (same as Model 1)")
assert len(FX_A) == len(FX_B) - len(s03.DEFAULT_DEGRADED_COLS)
print("OK: Variant A is strictly narrower, Variant B matches Model 1's full feature count.")


Variant A (removal) feature count: 10 fx_ columns
Variant B (resample) feature count: 34 fx_ columns (same as Model 1)
OK: Variant A is strictly narrower, Variant B matches Model 1's full feature count.


In [4]:
# Verification: pre-anchor-only climatology (no leakage) -- spot check one column, one tower, one anchor
dv2 = pd.read_csv(f"{s03.HOURLY}/forecast_daily_v2.csv", low_memory=False)
dv2["Datetime"] = pd.to_datetime(dv2["Datetime"], format="mixed")
dft4 = dv2[dv2.tower == 4].set_index("Datetime").sort_index()
anchor = pd.Timestamp("2021-12-16")
target_dates = pd.date_range(anchor + pd.Timedelta(days=1), periods=365, freq="D")
hist = dft4.loc[:anchor, "fx_SWC_mean"].dropna()
print("Pre-anchor history max date:", hist.index.max(), "<= anchor:", anchor, "->", hist.index.max() <= anchor)


Pre-anchor history max date: 2021-12-16 00:00:00 <= anchor: 2021-12-16 00:00:00 -> True


## Full sweep results

The full 3-tower x 5-anchor x 2-variant sweep was run via
`notebooks/07_scenario_analysis/s03_driver_availability_ablation.py` (not re-executed here -- ~10-15 min,
matches the B-10/B-15 script-does-the-heavy-lifting pattern). Loading its output and Model 1's existing D-65
numbers below.

In [5]:
summary = pd.read_csv(f"{s03.RESULTS}/s03_summary.csv")
summary_gf = pd.read_csv(f"{s03.RESULTS}/s03_summary_vs_gapfilled.csv")
print(f"s03_summary.csv: {len(summary)} rows, towers={sorted(summary.tower.unique())}, "
      f"anchors={sorted(summary.anchor_year.unique())}, variants={sorted(summary.variant.unique())}, "
      f"models={sorted(summary.model.unique())}")
assert sorted(summary.tower.unique()) == [2, 4, 9], "full 3-tower coverage check"
assert sorted(summary.anchor_year.unique()) == [2018, 2019, 2020, 2021, 2022], "full 5-anchor coverage check"


s03_summary.csv: 1980 rows, towers=[np.int64(2), np.int64(4), np.int64(9)], anchors=[np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)], variants=['A_removal', 'B_resample'], models=['DLinear', 'Ensemble_MASEweighted', 'Ensemble_unweighted', 'LSTM', 'LightGBM', 'RF', 'SARIMAX', 'TFT', 'TabICLv2', 'TabPFN', 'XGB']


In [6]:
import compile_s03_results as cs03
cs03.main()


=== Observed target ===


[OK] Saved c:\Users\Nicholas\Documents\COMP0191 MSc Artificial Intelligence for Sustainable Development Project\results/s03_table_all_towers.csv


[OK] Saved c:\Users\Nicholas\Documents\COMP0191 MSc Artificial Intelligence for Sustainable Development Project\results/s03_table_by_tower.csv

=== Gap-filled target (secondary, exploratory -- see circularity caveat) ===


[OK] Saved c:\Users\Nicholas\Documents\COMP0191 MSc Artificial Intelligence for Sustainable Development Project\results/s03_table_vs_gapfilled_all_towers.csv


[OK] Saved c:\Users\Nicholas\Documents\COMP0191 MSc Artificial Intelligence for Sustainable Development Project\results/s03_table_vs_gapfilled_by_tower.csv

=== Combining into primary (headline) tables ===
[OK] Saved combined (Observed+GapFilled) c:\Users\Nicholas\Documents\COMP0191 MSc Artificial Intelligence for Sustainable Development Project\results/s03_table_all_towers.csv
[OK] Saved combined (Observed+GapFilled) c:\Users\Nicholas\Documents\COMP0191 MSc Artificial Intelligence for Sustainable Development Project\results/s03_table_by_tower.csv


In [7]:
pd.set_option("display.width", 220)
table_all = pd.read_csv(f"{s03.RESULTS}/s03_table_all_towers.csv", index_col=0, header=[0, 1, 2])
print("=== All-tower pooled: Observed + GapFilled x Model1/VariantA/VariantB, MASE/R2 ===")
print(table_all.loc[:, (slice(None), slice(None), ["MASE", "R2"])])


=== All-tower pooled: Observed + GapFilled x Model1/VariantA/VariantB, MASE/R2 ===
                      Observed                                    GapFilled                                    Observed                                    GapFilled                                   
                        Model1 VariantA_removal VariantB_resample    Model1 VariantA_removal VariantB_resample   Model1 VariantA_removal VariantB_resample    Model1 VariantA_removal VariantB_resample
                          MASE             MASE              MASE      MASE             MASE              MASE       R2               R2                R2        R2               R2                R2
model                                                                                                                                                                                                  
RF                       0.968            1.001             0.943     0.800            0.912             0.798   -0.2

In [8]:
table_by_tower = pd.read_csv(f"{s03.RESULTS}/s03_table_by_tower.csv", index_col=[0, 1], header=[0, 1, 2])
print("=== Per-tower (anchors averaged): Observed + GapFilled x Model1/VariantA/VariantB, MASE/R2 ===")
print(table_by_tower.loc[:, (slice(None), slice(None), ["MASE", "R2"])])


=== Per-tower (anchors averaged): Observed + GapFilled x Model1/VariantA/VariantB, MASE/R2 ===
                            Observed                                    GapFilled                                    Observed                                    GapFilled                                   
                              Model1 VariantA_removal VariantB_resample    Model1 VariantA_removal VariantB_resample   Model1 VariantA_removal VariantB_resample    Model1 VariantA_removal VariantB_resample
                                MASE             MASE              MASE      MASE             MASE              MASE       R2               R2                R2        R2               R2                R2
tower model                                                                                                                                                                                                  
2     RF                       0.346            0.367             0.330     0.689

## Verdict

*(Filled in from the printed tables above once the full sweep completes -- MASE-first per CLAUDE.md's
standing convention, R2 rightmost. States plainly whether removal or resample costs more, and by how much,
relative to Model 1's real numbers.)*


## Update (D-8x, 2026-08-08): climatology baseline, TabICL-sourced data, full 11-model roster

Three changes requested to bring this notebook up to speed with later project conventions that
postdate S-03's original run (D-70, 2026-07-14) and its model-roster-extension addendum (also
2026-07-15, never wired into *this* notebook -- it only ever ran as a standalone script):

1. **MASE baseline -> day-of-year climatology** (D-71/D-80, 2026-08-02, supersedes D-37's
   chain-persistence convention project-wide). `s03_driver_availability_ablation.py` and
   `s03_model_roster_extension.py` were both edited: MASE/RMSSE now score against
   `climatology_baseline()` (climatology computed from strictly pre-anchor real `y_observed`, same
   recipe as `b10_b13_climatology_baseline.py`), not `rr.chain_persistence()`. The persistence
   series is still saved in every chain file for reference, just no longer used as MASE's
   denominator.

2. **Latest data: TabICL-sourced gap-filling** (D-79/D-80). `forecast_daily_v2_tabicl.csv` is a
   schema-identical sibling of `forecast_daily_v2.csv` -- confirmed by direct diff, only
   `y_gapfilled`/`ar_ch4_*` differ, every `fx_` driver and `y_observed` are byte-identical. Both
   scripts gained a `daily_csv` parameter so Variant A/B can be rerun on this newer data.

3. **Full 11-model roster, wired into this notebook** (previously only ever loaded by a standalone
   script, never by `S03_driver_availability_ablation.ipynb` itself).

**A design tension resolved along the way**: Model 1 was originally read from an existing,
never-rerun table (D-65, RF-sourced, persistence-scored). Swapping only Variant A/B to
TabICL-sourced data while leaving Model 1 on the old table would reintroduce exactly the
data-source/feature-availability conflation S-03 exists to avoid. Fixed by recomputing Model 1
too, wherever a TabICL-sourced daily file makes that possible -- via the SAME code path used for
the real ablation, with `remove_cols=[]`/`resample_cols=[]`/`degraded_cols=[]` (a fully real,
undegraded feature set, so Model 1 and Variant A/B are guaranteed comparable, not
separately-written "similar" computations). TFT/DLinear/LSTM read the hourly
`forecast_features_v2.csv`, which has **no TabICL-sourced sibling anywhere in this project**
(`build_forecasting_matrix_v2_tabicl.py`'s own docstring: "forecast_features_v2.csv (hourly) does
not depend on the gap-filled CH4 series at all") -- these 3 stay RF-sourced throughout (Model 1
*and* Variant A/B), an unavoidable, explicitly-stated data-availability limit, not a gap
introduced here. Their Variant A/B predictions are unaffected by either change (climatology is a
scoring-only change; there is no TabICL data to swap to), so they were rescored from the
already-saved chains rather than retrained from scratch for a metric-only update.

All of this runs via a new committed script, `s03_climatology_tabicl_update.py` (not re-executed
in this notebook -- ~21 min total, matches the existing script-does-the-heavy-lifting pattern). It
performs 6 steps: (1) tree/SARIMAX/ensemble Variant A/B on TabICL data, (2) the same
architecture's Model-1-equivalent on TabICL data, (3-4) the same pair for TabPFN/TabICLv2, (5)
climatology-rescoring of the existing DL chains (no retrain), (6) assembling the full 11-model
Model1/VariantA/VariantB tables for both targets. Output:
`results/s03_table_all_towers_climatology_tabicl.csv` / `s03_table_by_tower_climatology_tabicl.csv`.

In [9]:
table_all_v2 = pd.read_csv(f"{s03.RESULTS}/s03_table_all_towers_climatology_tabicl.csv",
                            index_col=0, header=[0, 1, 2])
print("=== All-tower pooled (climatology MASE, TabICL-sourced where possible): "
      "Observed + GapFilled x Model1/VariantA/VariantB ===")
print(table_all_v2.loc[:, (slice(None), slice(None), ["MASE", "R2"])])

=== All-tower pooled (climatology MASE, TabICL-sourced where possible): Observed + GapFilled x Model1/VariantA/VariantB ===
                      Observed                                    GapFilled                                    Observed                                    GapFilled                                   
                        Model1 VariantA_removal VariantB_resample    Model1 VariantA_removal VariantB_resample   Model1 VariantA_removal VariantB_resample    Model1 VariantA_removal VariantB_resample
                          MASE             MASE              MASE      MASE             MASE              MASE       R2               R2                R2        R2               R2                R2
model                                                                                                                                                                                                  
RF                       2.270            2.037             1.814     1.990 

In [10]:
table_bt_v2 = pd.read_csv(f"{s03.RESULTS}/s03_table_by_tower_climatology_tabicl.csv",
                           index_col=[0, 1], header=[0, 1, 2])
print("=== Per-tower (climatology MASE, TabICL-sourced where possible), observed target ===")
print(table_bt_v2.loc[:, ("Observed", slice(None), ["MASE"])])

# Sanity check: is TabICL-sourced y_gapfilled a materially different SERIES (not just a scoring
# artefact) from RF-sourced y_gapfilled? Direct comparison of the two gap-fillers' mean level vs.
# the real y_observed mean, per tower -- explains the large absolute-MASE shift seen above for the
# 6 models that TRAIN DIRECTLY on y_gapfilled (RF/XGB/LightGBM/SARIMAX/ensembles), vs. the much
# smaller shift for TabPFN/TabICLv2 (context is always y_observed; y_gapfilled only touches their
# fx_/secondary-metric columns) and zero shift for TFT/DLinear/LSTM (no TabICL data available).
dv_rf = pd.read_csv(f"{s03.HOURLY}/forecast_daily_v2.csv", low_memory=False)
dv_ic = pd.read_csv(f"{s03.HOURLY}/forecast_daily_v2_tabicl.csv", low_memory=False)
print("\n=== y_gapfilled mean level: RF-sourced vs. TabICL-sourced vs. real y_observed ===")
for t in [2, 4, 9]:
    obs_mean = dv_rf[dv_rf.tower == t]["y_observed"].mean()
    rf_mean = dv_rf[dv_rf.tower == t]["y_gapfilled"].mean()
    ic_mean = dv_ic[dv_ic.tower == t]["y_gapfilled"].mean()
    print(f"  Tower {t}: y_observed={obs_mean:6.2f}   RF-sourced y_gapfilled={rf_mean:6.2f}   "
          f"TabICL-sourced y_gapfilled={ic_mean:6.2f}")

=== Per-tower (climatology MASE, TabICL-sourced where possible), observed target ===
                            Observed                                   
                              Model1 VariantA_removal VariantB_resample
                                MASE             MASE              MASE
tower model                                                            
2     RF                       1.770            1.562             1.263
      XGB                      0.629            0.598             0.587
      LightGBM                 1.368            1.427             1.319
      SARIMAX                  1.357            0.988             1.351
      Ensemble_unweighted      1.247            1.103             1.098
      Ensemble_MASEweighted    1.238            1.099             1.091
      TFT                      0.801            0.522             0.790
      TabPFN                   0.444            0.444             0.483
      DLinear                  1.125            0.6

## Verdict (D-8x update)

**The original driver-availability finding replicates, qualitatively unchanged, under both
the climatology-baseline switch and the TabICL-sourced data.** For every one of the 11 models,
Variant B (resample) beats or is within noise of Model 1 on MASE, and the removal/resample
ordering is the same as the original persistence-scored, RF-sourced run (TFT remains the one
genuine reversal; TabICLv2 remains a close wash). Driver degradation itself still does not cost
material accuracy -- that headline is robust to both changes tested here.

**But a new, unplanned finding surfaced along the way, and it matters more than the two requested
changes: TabICL-sourced gap-filling is a bad target source for the tree/SARIMAX/ensemble family
specifically, well beyond what D-80's earlier (softer) finding suggested.** D-80 found TabICL
context makes TabPFN/TabICLv2 forecasting modestly worse (MASE +0.05 to +0.10). Here, RF/XGB/
LightGBM/SARIMAX/the 2 ensembles -- which train directly ON `y_gapfilled` as their fitting target,
not just condition on it -- show a much larger absolute MASE increase (roughly 1.3-2.5x the
original RF-sourced numbers) when switched to TabICL-sourced data, at all 3 towers, not just the
ones with sparse real coverage. The mechanism, confirmed directly in the cell above: TabICL's
gap-filled CH4 series sits at a substantially different mean level than RF's at every tower (most
dramatically at Tower 2 and Tower 9), and training a regression target directly on a
differently-calibrated series propagates that miscalibration into every prediction -- a much
larger effect than merely using it as historical *context* for a zero-shot foundation model.
TabPFN/TabICLv2 (context is always real `y_observed`; `y_gapfilled` only touches their `fx_`/
secondary-metric columns here) and TFT/DLinear/LSTM (no TabICL data available at all) are
correspondingly far less affected or entirely unaffected.

**Practical implication**: this independently confirms and *extends* D-80's own conclusion ("the
standing forecasting champion is unchanged -- D-79's better gap-filling is a real, useful result
for gap-filling itself, but does not transfer to forecasting"). It now also does not transfer to
the tree/SARIMAX/ensemble family, and the failure mode there is considerably more severe than the
foundation-model-context case D-80 already flagged. **RF-sourced gap-filling remains the right
choice for every model family this project forecasts with** -- this update does not change that
recommendation, but sharpens the reason why for the tree/SARIMAX/ensemble family specifically.